# 실습 A. 변경 영향 분석 (약 40분)

코드를 고치면 무엇을 다시 확인해야 하는지 알아야 합니다. 이 노트북에서는 영향 분석 도구로 바뀐 줄에서 나온 값이 어디까지 흘러가는지 따라가 봅니다.

- **A-1.** 영향 보고서를 읽습니다. 도구는 놓치지 않으려고 넓게 잡으므로, 괜히 들어간 테스트를 AI로 걸러 냅니다.
- **A-2.** 도구가 찾아 주는 것(코드가 그대로인 곳의 영향)과 못 찾는 것(라이브러리가 대신 부르는 코드)을 사례 하나씩으로 봅니다.

> 부품 이름이 낯설면 0_시작하기에서 본 구조 지도를 다시 띄웁니다. `lab.structure("queue")`처럼 부품부터 열 수도 있습니다: `"queue"`(대기열), `"disp"`(접수·꺼내기), `"watch"`(점검·복구), `"power"`(전력 관리), `"devices"`(기기 목록), `"hub"`(입력 파일과 상태 한 줄 읽기).

In [1]:
import lab   # 실습 도우미: lab.run(...)이 make 명령을 실행하고 결과 보고서를 아래에 보여 줍니다

## A-1. 변경 영향 보고서 읽기 (15분)

코드를 고치면 무엇을 다시 확인해야 할까요? 영향 분석 도구는 **바뀐 줄에서 계산한 값이 어디까지 흘러가는지**(def-use: 값을 만드는 곳 → 쓰는 곳)를 끝까지 따라갑니다.

- 바뀐 줄에서 계산한 값 → 그 값을 저장한 필드·배열·컬렉션 → 그것을 읽는 줄
- 그 값으로 계산한 반환값 → 그 반환값을 받아 쓰는 줄
- 그 값으로 판단하는 `if`·반복문 → 그 안에서 일어나는 모든 일(필드 쓰기, 메서드 호출)

메서드를 넘나들 때는 **부른 곳으로만 돌아가도록** 짝을 맞춥니다. 예를 들어 `size()`를 A와 B가 부를 때, A가 넘긴 값 때문에 달라진 반환값이 B에게로 가지 않습니다.

**대상: 변경 요청 2.** 플랫폼팀이 "허브 칩에서 나눗셈(`%`)이 느리니, 대기열 `RingEventQueue`에서 `%`를 없애 달라. 동작은 바꾸지 말 것"이라고 요청했고, AI가 `offer`, `poll`, `clear`, `snapshot` 네 메서드를 고쳤습니다. 먼저 무엇이 바뀌었는지 봅니다.

In [2]:
lab.run("show", V="scenarios/req2/ai")

$ make show V=scenarios/req2/ai


이제 영향 분석을 돌립니다.

In [3]:
lab.run("impact", V="scenarios/req2/ai")

$ make impact V=scenarios/req2/ai


**보고서 읽는 법**

| 항목 | 이 결과 | 뜻 |
|---|---|---|
| 바뀐 코드 | offer, poll, clear, snapshot | 분석의 출발점. 주석·줄바꿈만 바뀐 메서드는 뺍니다 |
| 패키지별 범위 | 대기열에서 시작해 기기·전력·로그까지 | 대기열은 허브의 한가운데 있어서 넓게 퍼집니다 |
| 데이터 | 24개 | 값이 달라질 수 있는 필드와 컬렉션 내용 |
| 메서드 | 79개 | 반환값, 예외, 필드에 쓰는 값 중 하나라도 달라질 수 있는 메서드. 펼치면 이유가 코드 줄로 보입니다 |
| 테스트 | 38개 중 33개 | 결과가 달라질 수 있는 테스트 |

특정 메서드가 왜 들어갔는지는 `WHY=`로 물어볼 수 있습니다. 그러면 전체 보고서 대신 그 메서드의 이유만 보여 줍니다. **위에서 아래로** 읽습니다. 노란 점이 바뀐 줄이고, 그 아래로 값이 지나간 코드 줄이 이어집니다. 줄 사이의 회색 글은 값이 어떻게 넘어갔는지입니다. 서로 다른 바뀐 메서드에서 출발하는 이유가 있으면 "다른 이유"로 3가지까지 보여 줍니다.

In [4]:
lab.run("impact", V="scenarios/req2/ai", WHY="power.PowerBudget#tryStart")

$ make impact V=scenarios/req2/ai WHY=power.PowerBudget#tryStart


첫 번째 이유: poll의 바뀐 줄 `head++`가 필드 `head`를 바꿉니다 → poll이 꺼내는 이벤트(44줄)가 달라질 수 있습니다 → dispatchOne이 꺼낸 이벤트가 `null`인지에 따라(102줄) 처리기를 부를지가 달라집니다 → DeviceStatusHandler가 `tryStart`를 부를지(54줄) → 승인 목록 `running`에 쓰는 값(32줄)이 달라질 수 있습니다.

### 도구가 고른 테스트를 AI로 걸러 내기

도구는 다시 돌릴 테스트로 38개 중 **33개**를 골랐습니다. 놓치지 않으려고 넓게 잡기 때문에, 이 중에는 바뀐 코드를 전혀 거치지 않는 테스트도 섞여 있습니다. 그런 테스트 하나를 같이 보고, AI에게 걸러 달라고 해 봅니다.

**① 바뀐 코드.** 변경 요청 2는 `RingEventQueue.java` 한 파일만 고쳤습니다.

In [5]:
lab.diff("legacy/src/main/java/com/dxlab/homehub/queue/RingEventQueue.java", "scenarios/req2/ai/src/main/java/com/dxlab/homehub/queue/RingEventQueue.java",
         title="변경 요청 2: RingEventQueue.java (왼쪽 원래 코드, 오른쪽 AI 패치)")

**② 도구가 고른 테스트 중 하나: `EventParserTest#testBasicLine`.** 아래가 이 테스트의 전부입니다. 글자 한 줄을 이벤트로 바꾸고 번호(`seq`)·기기·종류가 맞는지 확인할 뿐, **대기열은 만들지도 쓰지도 않습니다.** 대기열 코드가 바뀌어도 이 테스트의 결과는 달라질 수 없습니다.

In [6]:
lab.show("legacy/src/test/java/com/dxlab/homehub/EventParserTest.java", 10, 16, mark=[13],
         title="EventParserTest: 대기열이 나오지 않음")

**③ 그런데 도구는 왜 이 테스트를 넣었을까요?** 도구는 `Event`의 `seq` 칸을 **이벤트 객체마다 따로 보지 않고, 하나의 칸으로** 봅니다.

| | 실제로 일어나는 일 | 도구가 보는 것 |
|---|---|---|
| 쓰는 곳 | 대기열이 꺼내 준 이벤트를 보고, 규칙 검사가 과열 이벤트를 **새로 만들며** `seq`를 씀 (`RuleEngine` 48줄 `new Event(nextSeq++, ...)`) | "`Event.seq`에 쓰는 값이 달라질 수 있다" |
| 읽는 곳 | 테스트는 **자기가 만든 다른 이벤트**의 `seq`를 읽음 (위 13줄 `e.getSeq()`) | "`Event.seq`를 읽으니 영향받는다" |
| 결론 | 서로 다른 객체라 **상관없음** | **영향 있음** → 다시 돌릴 테스트에 넣음 |

도구는 하나도 놓치지 않는 쪽을 택해서 이렇게 "괜히 들어간" 테스트가 생깁니다. 도구가 따라간 경로는 `lab.run("impact", V="scenarios/req2/ai", WHY="EventParserTest#testBasicLine")`로 직접 볼 수 있습니다.

**④ AI에게 걸러 달라고 하기.** 도구가 고른 33개 목록, diff, 테스트 코드(약 400줄)를 AI에게 주고, 테스트마다 바뀐 코드를 실제로 거치는지 판단해 달라고 합니다(프롬프트 P-4). 실행하는 데 수십 초 걸립니다.

In [7]:
ai = lab.codex("P-4", diff=lab.diff_text("scenarios/req2/ai"),
               테스트_목록=lab.impact_tests("scenarios/req2/ai"),      # 도구가 고른 33개
               테스트_코드=lab.source_text("legacy/src/test"))          # 테스트 코드 전체

Codex 실행 중…
끝났습니다 (20초).


**⑤ 볼 것.** `EventParserTest`가 "뺌"으로 나왔는지, 이유가 ③과 같은지 봅니다. 이 환경에서 돌렸을 때는 33개 중 17개를 빼서 **16개**가 남았습니다.

**정리.**
- 영향 범위를 **빠짐없이** 찾는 일은 도구가 합니다. 대신 넓게 잡습니다.
- 넓게 잡힌 것 중 **괜히 들어간 것을 걸러 내는 일**은 AI가 도울 수 있습니다.
- AI가 뺀 것이 정말 빼도 되는지는 사람이 테스트 코드를 보고 확인합니다. AI도 틀릴 수 있습니다.

Codex를 쓸 수 없으면 아래 셀로 같은 프롬프트를 출력해 사내 AI 도구에 붙여 넣습니다.

In [ ]:
lab.prompt("P-4", diff=lab.diff_text("scenarios/req2/ai"),
           테스트_목록=lab.impact_tests("scenarios/req2/ai"), 테스트_코드=lab.source_text("legacy/src/test"))

## A-2. 도구가 찾아 주는 것과 못 찾는 것 (25분)

A-1에서는 도구가 넓게 잡는 경우를 봤습니다. 여기서는 두 가지를 사례 하나씩으로 봅니다.

1. **찾아 줌:** 코드가 한 줄도 바뀌지 않은 메서드라도, 바뀐 코드가 만든 값을 쓰면 영향받는다고 알려 줍니다.
2. **못 찾음:** 우리 코드가 아니라 라이브러리가 대신 부르는 코드는 따라가지 못합니다.

### ① 찾아 줌: 코드가 그대로인 Watchdog도 영향을 받는다

**변경 요청 1 (상품기획팀).** 저녁 시간에 기기 이벤트가 몰리면 대기열(8칸)이 가득 차서 이벤트를 거부합니다. "평소에는 지금처럼 8칸만 쓰고, 필요할 때만 최대 32칸까지 늘려 달라"는 요청입니다.

**AI 패치 A가 바꾼 것.** 대기열 `RingEventQueue.java` 한 파일만 고쳤습니다.
- 대기열이 가득 차면 새 메서드 `grow()`가 배열을 2배 크기로 바꿉니다(8 → 16 → 32칸).
- 그래서 배열 `buf`가 **실행 중에 바뀝니다.** 원래는 처음 만들 때 8칸으로 정해지고 끝이었습니다.

In [8]:
lab.diff("legacy/src/main/java/com/dxlab/homehub/queue/RingEventQueue.java",
         "scenarios/req1/ai-A/src/main/java/com/dxlab/homehub/queue/RingEventQueue.java",
         title="변경 요청 1: RingEventQueue.java (왼쪽 원래 코드, 오른쪽 AI 패치 A)")

**Watchdog은 고치지 않았습니다.** Watchdog은 처리가 막혔는지 감시하는 부품입니다. 대기열이 **75% 이상 찬 상태**가 여러 번 연속되면 "막힘" 경보를 남깁니다. 아래 22줄이 그 판단이고, 75%의 기준인 `queue.capacity()`는 대기열 배열의 크기(`buf.length`)를 돌려줍니다.

In [9]:
lab.show("legacy/src/main/java/com/dxlab/homehub/ops/Watchdog.java", 21, 33, mark=[22], title="Watchdog.check(): 패치 A에서 바뀌지 않은 코드")

**질문: 패치 A 때문에 `check()`의 결과가 달라질 수 있을까요?**

**답: 예, 달라집니다.** 패치 A에서는 배열이 8 → 16 → 32칸으로 커지므로 `capacity()`가 돌려주는 값이 바뀌고, 22줄의 "75%" 기준도 함께 바뀝니다. Watchdog 코드는 한 줄도 바뀌지 않았지만 동작은 바뀝니다. diff에 나오지 않으니 사람이 놓치기 쉬운 곳입니다.

도구가 이것을 찾아 주는지, `WHY=`로 `check()`만 물어봅니다.

In [10]:
lab.run("impact", V="scenarios/req1/ai-A", WHY="ops.Watchdog#check")

$ make impact V=scenarios/req1/ai-A WHY=ops.Watchdog#check


**도구는 찾아 줍니다.** 첫 번째 이유를 위에서 아래로 읽으면, 앞에서 설명한 것과 같은 길입니다.

`offer` 41줄 `grow()` → 필드 `buf`에 새 배열 저장 → `capacity()` 74줄 `return buf.length` → Watchdog 22줄 `if` 조건 → 23줄 막힘 횟수 → 28줄 경보 기록

실제로 어떻게 달라지는지 숫자로 보면 이렇습니다.

| 대기 중인 이벤트 | 원래 코드 (항상 8칸) | 패치 A |
|---|---|---|
| 6~8개 | 75% 이상 → 막힘으로 셈 (6×4 ≥ 8×3) | 같음 (아직 8칸) |
| 9~11개 | 8칸이 최대라 9개 이상 쌓일 수 없음 | 16칸으로 늘어 75% 미만 → **막힘으로 세지 않음** (9×4 < 16×3) |

**정리.** 도구는 코드가 바뀌지 않은 메서드도, 바뀐 코드가 만든 값을 쓰면 영향받는다고 찾아 줍니다. 다만 이 변화가 괜찮은지(이벤트 9~11개가 쌓여도 막힘으로 보지 않는 것이 맞는지)는 도구가 판단하지 않습니다. 사람이 판단합니다.

### ② 못 찾음: 라이브러리가 대신 부르는 코드

**요청 (앱 개발팀).** "상태 화면에서 기기 목록을 종류별로 묶어 보여 달라." 지금은 이름순(aircon-1, dryer-1, fridge-1, ...)입니다.

**AI가 바꾼 것.** `DeviceRegistry`의 정렬 비교 함수 하나를 "이름순"에서 "종류순, 같은 종류면 이름순"으로 바꿨습니다.

In [11]:
lab.run("show", V="scenarios/a3-sort")

$ make show V=scenarios/a3-sort


**이 순서는 어디에 쓰일까요?** `devices.all()`은 이 비교 함수로 정렬한 목록을 돌려줍니다(아래 53줄 `Collections.sort`). 그리고 두 곳이 이 순서를 그대로 씁니다.

In [12]:
lab.show("legacy/src/main/java/com/dxlab/homehub/device/DeviceRegistry.java", 42, 58, mark=[44, 53, 55], title="DeviceRegistry: all()은 sortById로 정렬해 돌려줌")

In [13]:
lab.show("legacy/src/main/java/com/dxlab/homehub/app/HomeHub.java", 139, 145, mark=[140], title="① 상태 출력 HomeHub.observable: 기기 상태를 이 순서대로 적음")

In [14]:
lab.show("legacy/src/main/java/com/dxlab/homehub/handlers/SafetyHandler.java", 34, 45, mark=[36, 40, 41], title="② 누수 때 SafetyHandler: 이 순서대로 물 쓰는 기기를 끔")

요청한 사람은 화면 순서만 바뀐다고 생각했지만, **누수 때 기기를 끄는 순서도 바뀝니다.** 도구는 이것을 찾아 줄까요?

In [15]:
lab.run("impact", V="scenarios/a3-sort")

$ make impact V=scenarios/a3-sort


**도구는 못 찾습니다.** 출력이 달라질 수 있는 메서드는 비교 함수 `compare` 2개뿐이고, 다시 돌릴 테스트는 **0개**입니다.

`compare`가 2개인 이유: 우리가 고친 `compare(Device, Device)` 말고, 컴파일러가 자동으로 만든 `compare(Object, Object)`가 하나 더 있습니다. `Comparator<Device>`를 구현하면 javac가 `return compare((Device) a, (Device) b);` 한 줄짜리 메서드(다리 메서드)를 만들어 줍니다. `Collections.sort`가 실제로 부르는 것은 이 메서드입니다. 즉 바뀐 값은 JDK가 부르는 입구까지 오고, 거기서 끊깁니다.

**왜 못 찾나요.** 비교 함수를 부르는 것은 우리 코드가 아니라 JDK의 `Collections.sort`입니다(위 53줄). 도구는 JDK 안을 분석하지 않으므로 "정렬된 목록의 순서는 `compare`의 반환값으로 정해진다"는 사실을 모릅니다. 그래서 값의 흐름이 `compare`에서 끊깁니다.

**그럼 어떻게 확인하나요.** 먼저 테스트를 전부 돌려 봅니다.

In [16]:
lab.run("variant-test", V="scenarios/a3-sort")

$ make variant-test V=scenarios/a3-sort
tests: 38, passed: 38, failed: 0


테스트는 38개 모두 통과합니다. 기기 순서를 확인하는 테스트가 없기 때문입니다.

남은 방법은 **동작 기록 비교**입니다. `golden/` 폴더에는 레거시 HomeHub의 동작 기록이 있습니다. 가전이 보낸 메시지를 모은 입력 파일 3개(`legacy/workloads/`)를 레거시에 넣고, 메시지 한 줄을 처리할 때마다 허브 상태(기기 목록 포함)를 한 줄씩 적어 둔 것입니다. 바꾼 코드에 같은 입력을 넣고, 나온 상태를 이 기록과 한 줄씩 비교합니다.

In [17]:
lab.run("trace", TRACE_CP="work/a3-sort/build/classes", TRACE_NAME="trace-a3-sort")

$ make trace TRACE_CP=work/a3-sort/build/classes TRACE_NAME=trace-a3-sort
기록 normal_day       다름 (29줄)
기록 burst            다름 (22줄)
기록 fridge_firmware  다름 (19줄)


**동작 기록은 다릅니다.** 세 파일 모두 처음부터 다르고, 표의 "기기 목록 순서" 항목을 보면 이름순(aircon-1 → dryer-1 → …)이 종류순(washer-1 → dryer-1 → …)으로 바뀌었습니다. 도구는 "영향 없음"이라고 했지만 실제 동작은 바뀌었습니다.

**정리.** 도구는 라이브러리가 대신 부르는 코드(콜백)를 따라가지 못합니다. 도구가 영향이 없다고 해도, 실제로 돌려 보는 동작 기록 비교로 확인해야 합니다.

**생각해 볼 것.**
- ①에서 이벤트 9~11개가 쌓여도 막힘으로 보지 않게 된 것은 괜찮을까요? 누가 무엇을 보고 판단해야 합니까?
- ②의 비교 함수 변경을 도구가 찾으려면 무엇을 더 알아야 합니까?